Imports, podem alterar conforme necessário

In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder

#### Lendo base de dados e removendo valores aleatoriamente **executar SEM ALTERAR**

In [2]:
#Carregando base de dados
penguins = sns.load_dataset('penguins')

np.random.seed(42)  #Definindo seed do random para replicabilidade

#Removendo valores
removidos = set()
porcentagem = 0.30 #Porcentagem (0~1) das células a serem removidas
qtdCelulas = len(penguins)*(len(penguins.columns)) #Quantidade de células na base de dados, ignorando a última coluna

for i in range(int(np.ceil(porcentagem*qtdCelulas))):
  linha = np.random.randint(0, len(penguins))
  coluna = np.random.randint(0, len(penguins.columns))
  while (linha, coluna) in removidos:
    linha = np.random.randint(0, len(penguins))
    coluna = np.random.randint(0, len(penguins.columns))

  penguins.iloc[linha, coluna] = np.nan
  removidos.add((linha,coluna))

penguins.info()
print("\n\nForam removidas ",str(len(removidos)), "células das ",str(qtdCelulas))
del removidos
del qtdCelulas
del porcentagem

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            232 non-null    object 
 1   island             241 non-null    object 
 2   bill_length_mm     233 non-null    float64
 3   bill_depth_mm      244 non-null    float64
 4   flipper_length_mm  247 non-null    float64
 5   body_mass_g        240 non-null    float64
 6   sex                235 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


Foram removidas  723 células das  2408


# **Podem modificar daqui para baixo para fazer a imputação de valores**

In [3]:
df = pd.DataFrame(penguins)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,NaN,18.7,NaN,3750.0,Male
1,NaN,NaN,39.5,17.4,186.0,NaN,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,NaN,NaN,NaN,NaN,NaN,NaN
4,Adelie,NaN,NaN,19.3,193.0,3450.0,NaN
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,NaN,46.8,14.3,215.0,NaN,NaN
341,NaN,Biscoe,NaN,NaN,NaN,5750.0,NaN
342,Gentoo,Biscoe,45.2,14.8,NaN,5200.0,Female


In [4]:
Nan_totais = penguins.isnull().sum().sum()
Nan_totais

np.int64(736)

In [5]:
# Quantas linhas tem pelo meno um dado faltante
df["com_missing"] = df.isnull().any(axis=1)
df["com_missing"].sum()

np.int64(312)

In [6]:
df.to_csv('penguins.csv', index=False, encoding='utf-8')

In [7]:
#KNNImputer não aceita string
#colunas (string) -> species, island, sex

colunas_numericas = ['bill_length_mm', 'bill_depth_mm',
                     'flipper_length_mm', 'body_mass_g']

colunas_categoricas = ['species', 'island', 'sex']

In [8]:
# Inicializa o DataFrame imputado (a imputação completa é feita na próxima célula)
penguins_imputado = penguins.copy()


# Imputação UNIFICADA: numéricas + categóricas em um único KNN, com normalização
### Parâmetros: n_neighbors=20, weights='uniform'
 - n_neighbors=20, para ter uma melhor resposta, já que existem muitos Nan
 depois que você converte categoria em dummy (0 e 1), as distâncias entre vizinhos ficam menos confiáveis por isso é melhor usar peso igual para todos os 20 vizinhos(uniform)

 - normalização: evita que body_mass (~2700-6300) domine sobre dummies (0-1)
 então usa a media e o desvio padrão para deixar os valores mais proximos de 0 evitando outliers

In [9]:
#cria as colunas binárias como float (0.0 e 1.0), necessário para o KNN
categoricas_encoded = pd.concat([
    pd.get_dummies(penguins[col], prefix=col, dtype=float).where(penguins[col].notna())
    for col in colunas_categoricas
], axis=1)

categoricas_encoded

,species_Adelie,species_Chinstrap,species_Gentoo,island_Biscoe,island_Dream,island_Torgersen,sex_Female,sex_Male
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
4,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
339,0.0,0.0,1.0,1.0,0.0,0.0,NaN,NaN
340,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN
341,NaN,NaN,NaN,1.0,0.0,0.0,NaN,NaN
342,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


In [10]:
# Normalização (z-score) ignorando NaN
mean_num = penguins[colunas_numericas].mean()
std_num = penguins[colunas_numericas].std()
numericas_norm = (penguins[colunas_numericas] - mean_num) / std_num

In [11]:
# Junta as numéricas normalizadas com as categoricas_encoded em um único DataFrame
features = pd.concat([numericas_norm, categoricas_encoded], axis=1)

In [12]:
# imputando usando os parametros ditos acima
imputer = KNNImputer(n_neighbors=20, weights='uniform')
features_imp = pd.DataFrame(imputer.fit_transform(features), columns=features.columns)

In [13]:
# Desnormalização: desfaz o z-score para voltar à escala original.
penguins_imputado[colunas_numericas] = features_imp[colunas_numericas] * std_num + mean_num
penguins_imputado

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,40.645,18.700,194.85,3750.00,Male
1,NaN,NaN,39.500,17.400,186.00,3356.25,Female
2,Adelie,Torgersen,40.300,18.000,195.00,3250.00,Female
3,Adelie,NaN,38.405,18.085,191.50,3765.00,NaN
4,Adelie,NaN,38.995,19.300,193.00,3450.00,NaN
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,45.070,15.465,215.70,4882.50,NaN
340,Gentoo,NaN,46.800,14.300,215.00,4896.25,NaN
341,NaN,Biscoe,43.885,15.545,213.40,5750.00,NaN
342,Gentoo,Biscoe,45.200,14.800,214.25,5200.00,Female


In [14]:
# Reconstrói as categorias via argmax (categoria com maior valor nas categoricas_encoded)
for col in colunas_categoricas:
    cols_da_cat = [c for c in categoricas_encoded.columns if c.startswith(col + '_')]
    penguins_imputado[col] = features_imp[cols_da_cat].idxmax(axis=1).str.replace(f'{col}_', '', regex=False)

penguins_imputado

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,40.645,18.700,194.85,3750.00,Male
1,Adelie,Dream,39.500,17.400,186.00,3356.25,Female
2,Adelie,Torgersen,40.300,18.000,195.00,3250.00,Female
3,Adelie,Dream,38.405,18.085,191.50,3765.00,Female
4,Adelie,Dream,38.995,19.300,193.00,3450.00,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,45.070,15.465,215.70,4882.50,Female
340,Gentoo,Biscoe,46.800,14.300,215.00,4896.25,Female
341,Gentoo,Biscoe,43.885,15.545,213.40,5750.00,Female
342,Gentoo,Biscoe,45.200,14.800,214.25,5200.00,Female


In [15]:
# verificando se deu certo (Nan restantes)
print("NaN restantes após imputação:")
print(penguins_imputado.isnull().sum())

NaN restantes após imputação:
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


In [16]:
# Preserva o flag original (quem TINHA missing antes da imputação)
penguins_imputado["com_missing"] = df["com_missing"].values
penguins_imputado["com_missing"]


,com_missing
0,True
1,True
2,False
3,True
4,True
...,...
339,True
340,True
341,True
342,True


In [17]:
penguins_imputado.to_csv('penguins_imputado.csv', index=False, encoding='utf-8')
penguins_imputado

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,com_missing
0,Adelie,Torgersen,40.645,18.700,194.85,3750.00,Male,True
1,Adelie,Dream,39.500,17.400,186.00,3356.25,Female,True
2,Adelie,Torgersen,40.300,18.000,195.00,3250.00,Female,False
3,Adelie,Dream,38.405,18.085,191.50,3765.00,Female,True
4,Adelie,Dream,38.995,19.300,193.00,3450.00,Female,True
...,...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,45.070,15.465,215.70,4882.50,Female,True
340,Gentoo,Biscoe,46.800,14.300,215.00,4896.25,Female,True
341,Gentoo,Biscoe,43.885,15.545,213.40,5750.00,Female,True
342,Gentoo,Biscoe,45.200,14.800,214.25,5200.00,Female,True


In [18]:
print(penguins_imputado.head(15))
print(penguins_imputado.isnull().sum())

   species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0   Adelie  Torgersen          40.645         18.700             194.85   
1   Adelie      Dream          39.500         17.400             186.00   
2   Adelie  Torgersen          40.300         18.000             195.00   
3   Adelie      Dream          38.405         18.085             191.50   
4   Adelie      Dream          38.995         19.300             193.00   
5   Adelie      Dream          41.030         20.600             190.00   
6   Adelie     Biscoe          38.790         17.800             181.00   
7   Adelie      Dream          39.200         18.105             195.00   
8   Adelie  Torgersen          34.100         18.100             193.00   
9   Adelie  Torgersen          42.000         20.200             190.00   
10  Adelie      Dream          37.800         17.100             186.00   
11  Adelie      Dream          38.270         17.300             180.00   
12  Adelie      Dream    

In [19]:
df_antes = df.copy()
df_antes["momento"] = "Antes"
df_antes

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,com_missing,momento
0,Adelie,Torgersen,NaN,18.7,NaN,3750.0,Male,True,Antes
1,NaN,NaN,39.5,17.4,186.0,NaN,Female,True,Antes
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,False,Antes
3,Adelie,NaN,NaN,NaN,NaN,NaN,NaN,True,Antes
4,Adelie,NaN,NaN,19.3,193.0,3450.0,NaN,True,Antes
...,...,...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,True,Antes
340,Gentoo,NaN,46.8,14.3,215.0,NaN,NaN,True,Antes
341,NaN,Biscoe,NaN,NaN,NaN,5750.0,NaN,True,Antes
342,Gentoo,Biscoe,45.2,14.8,NaN,5200.0,Female,True,Antes


In [20]:
df_depois = penguins_imputado.copy()
df_depois["momento"] = "Depois"
df_depois

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,com_missing,momento
0,Adelie,Torgersen,40.645,18.700,194.85,3750.00,Male,True,Depois
1,Adelie,Dream,39.500,17.400,186.00,3356.25,Female,True,Depois
2,Adelie,Torgersen,40.300,18.000,195.00,3250.00,Female,False,Depois
3,Adelie,Dream,38.405,18.085,191.50,3765.00,Female,True,Depois
4,Adelie,Dream,38.995,19.300,193.00,3450.00,Female,True,Depois
...,...,...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,45.070,15.465,215.70,4882.50,Female,True,Depois
340,Gentoo,Biscoe,46.800,14.300,215.00,4896.25,Female,True,Depois
341,Gentoo,Biscoe,43.885,15.545,213.40,5750.00,Female,True,Depois
342,Gentoo,Biscoe,45.200,14.800,214.25,5200.00,Female,True,Depois


In [21]:
penguins_unificado = pd.concat([df_antes, df_depois], ignore_index= True)
penguins_unificado.shape

(688, 9)

In [22]:
penguins_unificado.to_csv('penguins_unificado.csv', index=False, encoding='utf-8')
penguins_unificado

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,com_missing,momento
0,Adelie,Torgersen,NaN,18.700,NaN,3750.00,Male,True,Antes
1,NaN,NaN,39.500,17.400,186.00,NaN,Female,True,Antes
2,Adelie,Torgersen,40.300,18.000,195.00,3250.00,Female,False,Antes
3,Adelie,NaN,NaN,NaN,NaN,NaN,NaN,True,Antes
4,Adelie,NaN,NaN,19.300,193.00,3450.00,NaN,True,Antes
...,...,...,...,...,...,...,...,...,...
683,Gentoo,Biscoe,45.070,15.465,215.70,4882.50,Female,True,Depois
684,Gentoo,Biscoe,46.800,14.300,215.00,4896.25,Female,True,Depois
685,Gentoo,Biscoe,43.885,15.545,213.40,5750.00,Female,True,Depois
686,Gentoo,Biscoe,45.200,14.800,214.25,5200.00,Female,True,Depois


In [23]:
#Fazendo os kpi´s
colunas = colunas_categoricas+colunas_numericas

#Antes
total_antes = len(df_antes)
completos_antes = int((~df_antes['com_missing']).sum())
com_missing_antes = int(df_antes['com_missing'].sum())
pct_completos_antes = round(completos_antes / total_antes * 100, 2)

#Depois
total_depois = len(df_depois)
completos_depois = int((~df_depois[colunas].isnull().any(axis=1)).sum())
com_missing_depois = int(df_depois[colunas].isnull().any(axis=1).sum())
pct_completos_depois = round(completos_depois / total_depois * 100, 2)

kpis = pd.DataFrame({
'momento': ['Antes', 'Depois'],
'total_registros': [total_antes, total_depois],
'registros_completos': [completos_antes, completos_depois],
'registros_com_missing': [com_missing_antes, com_missing_depois],
'pct_completos': [pct_completos_antes/100, pct_completos_depois/100]
})
print(kpis)
kpis.to_csv('kpis.csv', index=False, encoding='utf-8')

  momento  total_registros  registros_completos  registros_com_missing  \
0   Antes              344                   32                    312   
1  Depois              344                  344                      0   

   pct_completos  
0          0.093  
1          1.000  


In [24]:
#Tabela da % de missing por variavel
missing_antes = df_antes[colunas].isnull().mean().round(4)
missing_depois = df_depois[colunas].isnull().mean().round(4)

missing_por_variavel = pd.concat([
pd.DataFrame({'variavel': missing_antes.index,
'percentual_missing': missing_antes.values,
'momento': 'Antes'}),
pd.DataFrame({'variavel': missing_depois.index,
'percentual_missing': missing_depois.values,
'momento': 'Depois'})
], ignore_index=True)

print(missing_por_variavel)
missing_por_variavel.to_csv('missing_por_variavel.csv', index=False, encoding='utf-8')

             variavel  percentual_missing momento
0             species              0.3256   Antes
1              island              0.2994   Antes
2                 sex              0.3169   Antes
3      bill_length_mm              0.3227   Antes
4       bill_depth_mm              0.2907   Antes
5   flipper_length_mm              0.2820   Antes
6         body_mass_g              0.3023   Antes
7             species              0.0000  Depois
8              island              0.0000  Depois
9                 sex              0.0000  Depois
10     bill_length_mm              0.0000  Depois
11      bill_depth_mm              0.0000  Depois
12  flipper_length_mm              0.0000  Depois
13        body_mass_g              0.0000  Depois
